# Dataset : CIFAR-10  
            60,000 colour images (32×32×3) across 10 object classes:
            airplane, automobile, bird, cat, deer,
            dog, frog, horse, ship, truck
  Libraries: NumPy · Pandas · Matplotlib · Seaborn · Scikit-Learn · TensorFlow

In [ ]:
# Imports
import os, time, warnings
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

import seaborn as sns
warnings.filterwarnings("ignore")


In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.decomposition   import PCA
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.pipeline        import Pipeline

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Output directory
OUT = "cifar10_outputs"
os.makedirs(OUT, exist_ok=True)

CLASS_NAMES = ["airplane","automobile","bird","cat","deer",
               "dog","frog","horse","ship","truck"]

In [ ]:
# Colour palette
BG_DARK  = "#0d0d1a"
BG_PANEL = "#16213e"
BG_MID   = "#1a1a2e"
ACCENT   = "#4C72B0"
GREEN    = "#2ecc71"
RED      = "#e74c3c"

def _style(fig, *axes):
    fig.patch.set_facecolor(BG_DARK)
    for ax in axes:
        ax.set_facecolor(BG_PANEL)
        ax.tick_params(colors="white")
        for sp in ax.spines.values():
            sp.set_edgecolor("#444466")

print("="*72)
print("  ML JOURNEY — CIFAR-10 Image Classification")
print(f"  TensorFlow {tf.__version__}  |  NumPy {np.__version__}  |  Pandas {pd.__version__}")
print("="*72)

results = {}   # name → {accuracy, train_time, history, type}




In [ ]:
#  DATA LOADING & EXPLORATORY DATA ANALYSIS
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.cifar10.load_data()
y_tr_raw = y_tr_raw.flatten()
y_te_raw = y_te_raw.flatten()

print(f"  Train : {X_tr_raw.shape}  |  Test : {X_te_raw.shape}")
print(f"  Pixel range : [{X_tr_raw.min()}, {X_tr_raw.max()}]")

In [ ]:
# Pandas EDA
flat = X_tr_raw.reshape(50_000, -1)          # 50000 × 3072
df_eda = pd.DataFrame({
    "label"   : y_tr_raw,
    "class"   : [CLASS_NAMES[i] for i in y_tr_raw],
    "mean_R"  : flat[:, 0:1024].mean(1),
    "mean_G"  : flat[:, 1024:2048].mean(1),
    "mean_B"  : flat[:, 2048:].mean(1),
    "brightness": flat.mean(1),
})
print("\n  Pandas head (first 5 rows):")
print(df_eda.head())
print("\n  Class distribution (training):")
print(df_eda["class"].value_counts().to_string())
print("\n  Channel statistics:")
print(df_eda[["mean_R","mean_G","mean_B","brightness"]].describe().round(2))


In [ ]:
# sample grid + class distribution
fig = plt.figure(figsize=(22, 11))
_style(fig)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Sample grid (3 × 10)
ax_grid = fig.add_subplot(gs[0, :2])
ax_grid.axis("off")
ax_grid.set_facecolor(BG_DARK)
ax_grid.set_title("Sample Images — 3 per Class", color="white", fontsize=14, pad=8)
for ci, cname in enumerate(CLASS_NAMES):
    idxs = np.where(y_tr_raw == ci)[0][:3]
    for j, idx in enumerate(idxs):
        ax_s = fig.add_axes([0.03 + ci*0.087, 0.58 - j*0.145, 0.075, 0.13])
        ax_s.imshow(X_tr_raw[idx])
        if j == 0:
            ax_s.set_title(cname, color="white", fontsize=7)
        ax_s.axis("off")
        
# Class distribution
ax_dist = fig.add_subplot(gs[0, 2])
_style(fig, ax_dist)
counts = df_eda["class"].value_counts().loc[CLASS_NAMES]
clrs   = sns.color_palette("plasma", 10)
ax_dist.barh(CLASS_NAMES, counts.values, color=clrs, edgecolor="white", lw=0.4)
ax_dist.set_title("Class Distribution", color="white", fontsize=13)
ax_dist.set_xlabel("Count", color="#aaaaaa")
ax_dist.tick_params(colors="white")
for v, bar in zip(counts.values, ax_dist.patches):
    ax_dist.text(bar.get_width()+30, bar.get_y()+bar.get_height()/2,
                 f"{v:,}", va="center", color="white", fontsize=8)


In [ ]:
# Channel histograms
ax_ch = fig.add_subplot(gs[1, :2])
_style(fig, ax_ch)
sample = df_eda.sample(8000, random_state=SEED)
ax_ch.hist(sample["mean_R"], bins=50, color="red",   alpha=0.6, label="Red",   density=True)
ax_ch.hist(sample["mean_G"], bins=50, color="green", alpha=0.6, label="Green", density=True)
ax_ch.hist(sample["mean_B"], bins=50, color="#4488ff",alpha=0.6,label="Blue",  density=True)
ax_ch.set_title("Per-Channel Mean Intensity (8k samples)", color="white", fontsize=13)
ax_ch.set_xlabel("Mean Pixel Value", color="#aaaaaa")
ax_ch.set_ylabel("Density", color="#aaaaaa")
ax_ch.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")


In [ ]:
# Brightness violin
ax_vio = fig.add_subplot(gs[1, 2])
_style(fig, ax_vio)
sns.violinplot(data=df_eda.sample(5000, random_state=SEED),
               x="class", y="brightness", order=CLASS_NAMES,
               palette="plasma", inner="quartile", ax=ax_vio)
ax_vio.set_title("Brightness Distribution by Class", color="white", fontsize=12)
ax_vio.set_xlabel("", color="#aaaaaa")
ax_vio.set_ylabel("Mean Brightness", color="#aaaaaa")
ax_vio.set_xticklabels(CLASS_NAMES, rotation=40, ha="right",
                        color="white", fontsize=8)
ax_vio.tick_params(colors="white")

fig.suptitle("CIFAR-10 — Exploratory Data Analysis",
             color="white", fontsize=18, fontweight="bold", y=0.99)
plt.savefig(f"{OUT}/01_eda.png", dpi=150, bbox_inches="tight",
            facecolor=BG_DARK)
plt.close()
print(f"\n  [✓] EDA plot → {OUT}/01_eda.png")


In [ ]:
# PCA + mean class images

print("  Computing PCA …")
pca_viz  = PCA(n_components=50, random_state=SEED)
flat_n   = flat[:10_000] / 255.0
X_pca_v  = pca_viz.fit_transform(flat_n)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
_style(fig, *axes)

In [ ]:
# Scree plot
expl = pca_viz.explained_variance_ratio_ * 100
axes[0].bar(range(1, 51), expl, color=ACCENT, alpha=0.8)
axes[0].plot(range(1, 51), np.cumsum(expl), "w-o",
             lw=2, ms=4, label="Cumulative")
axes[0].axhline(80, color=GREEN, ls="--", lw=1.5, label="80%")
axes[0].set_title("PCA Explained Variance", color="white", fontsize=13)
axes[0].set_xlabel("PC", color="#aaaaaa")
axes[0].set_ylabel("Variance %", color="#aaaaaa")
axes[0].legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")


In [ ]:
# PC1 vs PC2 scatter
labels_sub = y_tr_raw[:10_000]
for ci in range(10):
    m = labels_sub == ci
    axes[1].scatter(X_pca_v[m, 0], X_pca_v[m, 1],
                    s=5, alpha=0.4, label=CLASS_NAMES[ci])
axes[1].set_title("PCA — PC1 vs PC2", color="white", fontsize=13)
axes[1].set_xlabel("PC1", color="#aaaaaa")
axes[1].set_ylabel("PC2", color="#aaaaaa")
legend = axes[1].legend(markerscale=3, fontsize=7,
                         facecolor="#2a2a4a", edgecolor="white",
                         labelcolor="white", ncol=2)


In [ ]:
# Mean images
axes[2].axis("off")
axes[2].set_title("Mean Image per Class", color="white", fontsize=13)
for ci in range(10):
    mean_img = X_tr_raw[y_tr_raw == ci].mean(0).astype(np.uint8)
    ax_m = fig.add_axes([0.68 + (ci % 5)*0.058,
                          0.08 + (1 - ci//5)*0.38,
                          0.05, 0.3])
    ax_m.imshow(mean_img)
    ax_m.set_title(CLASS_NAMES[ci], color="white", fontsize=7)
    ax_m.axis("off")

fig.suptitle("CIFAR-10 — PCA & Mean Images",
             color="white", fontsize=16, fontweight="bold")
plt.savefig(f"{OUT}/02_pca.png", dpi=150, bbox_inches="tight",
            facecolor=BG_DARK)
plt.close()
print(f"  [✓] PCA plot → {OUT}/02_pca.png")

In [ ]:
# DATA PREPROCESSING
# Normalized float arrays 
X_tr_f  = X_tr_raw.astype("float32") / 255.0   # (50000, 32, 32, 3)
X_te_f  = X_te_raw.astype("float32") / 255.0   # (10000, 32, 32, 3)



In [ ]:
# Flattened (for sklearn)
X_tr_flat = X_tr_f.reshape(50_000, -1)          # (50000, 3072)
X_te_flat = X_te_f.reshape(10_000, -1)          # (10000, 3072)

In [ ]:
# One-hot labels 
y_tr_oh  = keras.utils.to_categorical(y_tr_raw, N_CLASSES)
y_te_oh  = keras.utils.to_categorical(y_te_raw, N_CLASSES)

In [ ]:
# Sklearn subset
SK_TR = 15_000;  SK_TE = 3_000
X_sk_tr = X_tr_flat[:SK_TR];  y_sk_tr = y_tr_raw[:SK_TR]
X_sk_te = X_te_flat[:SK_TE];  y_sk_te = y_te_raw[:SK_TE]

In [ ]:
# Channel-wise normalisation for Keras
ch_mean = X_tr_f.mean(axis=(0,1,2))        # shape (3,)
ch_std  = X_tr_f.std(axis=(0,1,2))         # shape (3,)
X_tr_n  = (X_tr_f - ch_mean) / (ch_std + 1e-7)
X_te_n  = (X_te_f - ch_mean) / (ch_std + 1e-7)
print(f"  Flat shape  : {X_sk_tr.shape}  |  pixel range [{X_sk_tr.min():.2f}, {X_sk_tr.max():.2f}]")
print(f"  Normalised  : mean≈{X_tr_n.mean():.4f}  std≈{X_tr_n.std():.4f}")



In [ ]:
# HELPER UTILITIES
def save_cm(cm, title, fpath, cmap="Blues"):
    fig, ax = plt.subplots(figsize=(11, 9))
    _style(fig, ax)
    cm_pct = cm.astype(float) / cm.sum(1, keepdims=True) * 100
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap=cmap,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.4, linecolor="#2a2a4a",
                cbar_kws={"label":"% True Class"}, ax=ax)
    ax.set_title(title, color="white", fontsize=14, pad=10)
    ax.set_xlabel("Predicted", color="#aaaaaa")
    ax.set_ylabel("True", color="#aaaaaa")
    ax.tick_params(colors="white", labelrotation=30)
    plt.tight_layout()
    plt.savefig(fpath, dpi=130, bbox_inches="tight", facecolor=BG_DARK)
    plt.close()

In [ ]:
def save_history(hist, title, fpath):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    _style(fig, *axes)
    for ax, metric, label in zip(axes, ["loss","accuracy"], ["Loss","Accuracy"]):
        ax.plot(hist.history[metric],        color=ACCENT,  lw=2.5, label="Train")
        ax.plot(hist.history[f"val_{metric}"],color=GREEN, lw=2.5, ls="--", label="Val")
        ax.set_title(f"{title} — {label}", color="white", fontsize=13)
        ax.set_xlabel("Epoch", color="#aaaaaa")
        ax.set_ylabel(label, color="#aaaaaa")
        ax.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")
    plt.tight_layout()
    plt.savefig(fpath, dpi=130, bbox_inches="tight", facecolor=BG_DARK)
    plt.close()

In [ ]:
def run_sklearn(name, model, Xtr, ytr, Xte, yte, pipe=False):
    print(f"\n  ▶ {name}")
    t0 = time.time()
    model.fit(Xtr, ytr)
    elapsed = time.time() - t0
    pred = model.predict(Xte)
    acc  = accuracy_score(yte, pred)
    results[name] = {"accuracy": acc, "train_time": elapsed,
                     "history": None, "type": "sklearn"}
    print(f"    Accuracy : {acc*100:.2f}%   Time : {elapsed:.1f}s")
    return pred, confusion_matrix(yte, pred)



In [ ]:
# BASELINE — LOGISTIC REGRESSION

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, C=1.0,
                                   solver="saga", multi_class="multinomial",
                                   random_state=SEED, n_jobs=-1))
])
lr_pred, lr_cm = run_sklearn("Logistic Regression", lr,
                              X_sk_tr, y_sk_tr, X_sk_te, y_sk_te)
save_cm(lr_cm, "Logistic Regression — Confusion Matrix",
        f"{OUT}/03_lr_cm.png", "Blues")

In [ ]:
# Per-class breakdown
lr_report = classification_report(y_sk_te, lr_pred,
                                   target_names=CLASS_NAMES, output_dict=True)
df_lr = pd.DataFrame(lr_report).T.loc[CLASS_NAMES,
                                       ["precision","recall","f1-score"]]
fig, ax = plt.subplots(figsize=(14, 5))
_style(fig, ax)
x, w = np.arange(10), 0.27
ax.bar(x-w,  df_lr["precision"].astype(float), w, label="Precision", color="#4C72B0")
ax.bar(x,    df_lr["recall"].astype(float),    w, label="Recall",    color="#DD8452")
ax.bar(x+w,  df_lr["f1-score"].astype(float),  w, label="F1",        color="#55A868")
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=30,
                                      ha="right", color="white", fontsize=9)
ax.set_title("Logistic Regression — Per-Class Metrics",
             color="white", fontsize=13)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score", color="#aaaaaa")
ax.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")
plt.tight_layout()
plt.savefig(f"{OUT}/04_lr_class_metrics.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()
print(f"  [✓] LR plots saved")


In [ ]:
# MODEL 2 — DECISION TREE
dt = DecisionTreeClassifier(max_depth=25, criterion="gini", random_state=SEED)
dt_pred, dt_cm = run_sklearn("Decision Tree (depth=25)", dt,
                              X_sk_tr, y_sk_tr, X_sk_te, y_sk_te)
save_cm(dt_cm, "Decision Tree — Confusion Matrix",
        f"{OUT}/05_dt_cm.png", "Purples")

# Depth vs accuracy
print("  Depth sensitivity …")
depths, daccs = [3,5,8,12,18,25,35], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    m.fit(X_sk_tr, y_sk_tr)
    daccs.append(accuracy_score(y_sk_te, m.predict(X_sk_te)))
    print(f"    depth={d:3d} → {daccs[-1]*100:.2f}%")
fig, ax = plt.subplots(figsize=(10, 5))
_style(fig, ax)
ax.plot(depths, [a*100 for a in daccs], "o-",
        color="#a29bfe", lw=2.5, ms=9)
ax.fill_between(depths, [a*100 for a in daccs], alpha=0.15, color="#a29bfe")
ax.set_title("Decision Tree — Depth vs Accuracy", color="white", fontsize=13)
ax.set_xlabel("max_depth", color="#aaaaaa")
ax.set_ylabel("Accuracy (%)", color="#aaaaaa")
plt.tight_layout()
plt.savefig(f"{OUT}/06_dt_depth.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()
print(f"  [✓] DT plots saved")



In [ ]:
# MODEL 3 — RANDOM FOREST
rf = RandomForestClassifier(n_estimators=300, max_depth=None,
                             min_samples_split=3, n_jobs=-1, random_state=SEED)
rf_pred, rf_cm = run_sklearn("Random Forest (300 trees)", rf,
                              X_sk_tr, y_sk_tr, X_sk_te, y_sk_te)
save_cm(rf_cm, "Random Forest — Confusion Matrix",
        f"{OUT}/07_rf_cm.png", "Greens")

# Feature importance as image (per channel)
imp = rf.feature_importances_.reshape(32, 32, 3)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
_style(fig, *axes)
ch_labels = ["Red channel", "Green channel", "Blue channel"]
for ci in range(3):
    im = axes[ci].imshow(imp[:,:,ci], cmap="hot")
    axes[ci].set_title(ch_labels[ci], color="white", fontsize=11)
    axes[ci].axis("off")
    plt.colorbar(im, ax=axes[ci]).ax.yaxis.label.set_color("white")
axes[3].imshow(imp.mean(-1), cmap="inferno")
axes[3].set_title("All-channel mean", color="white", fontsize=11)
axes[3].axis("off")
fig.suptitle("Random Forest — Pixel Importance Maps",
             color="white", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUT}/08_rf_importance.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()


In [ ]:
# Trees vs accuracy
print("  n_estimators sensitivity …")
n_trees_list, nta = [10, 50, 100, 200, 300], []
for nt in n_trees_list:
    m = RandomForestClassifier(n_estimators=nt, n_jobs=-1, random_state=SEED)
    m.fit(X_sk_tr, y_sk_tr)
    nta.append(accuracy_score(y_sk_te, m.predict(X_sk_te)))
    print(f"    trees={nt:4d} → {nta[-1]*100:.2f}%")
fig, ax = plt.subplots(figsize=(10, 5))
_style(fig, ax)
ax.plot(n_trees_list, [a*100 for a in nta], "o-",
        color=GREEN, lw=2.5, ms=9)
ax.fill_between(n_trees_list, [a*100 for a in nta], alpha=0.15, color=GREEN)
ax.set_title("Random Forest — n_estimators vs Accuracy",
             color="white", fontsize=13)
ax.set_xlabel("Number of Trees", color="#aaaaaa")
ax.set_ylabel("Accuracy (%)", color="#aaaaaa")
plt.tight_layout()
plt.savefig(f"{OUT}/09_rf_ntrees.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()
print(f"  [✓] RF plots saved")

In [ ]:
# MODEL 4 — SVM with HOG-like PCA features
SVM_N = 8_000
scaler_svm = StandardScaler()
pca_svm    = PCA(n_components=200, random_state=SEED)
Xsvm_tr    = pca_svm.fit_transform(scaler_svm.fit_transform(X_sk_tr[:SVM_N]))
Xsvm_te    = pca_svm.transform(scaler_svm.transform(X_sk_te))
y_svm_tr   = y_sk_tr[:SVM_N]

svm = SVC(kernel="rbf", C=10.0, gamma="scale",
          decision_function_shape="ovr", random_state=SEED)
print(f"\n  ▶ SVM (RBF, PCA-200 features, {SVM_N} samples)")
t0 = time.time()
svm.fit(Xsvm_tr, y_svm_tr)
elapsed_svm = time.time() - t0
svm_pred    = svm.predict(Xsvm_te)
svm_acc     = accuracy_score(y_sk_te, svm_pred)
results["SVM (RBF + PCA-200)"] = {"accuracy": svm_acc,
                                    "train_time": elapsed_svm,
                                    "history": None, "type": "sklearn"}
svm_cm = confusion_matrix(y_sk_te, svm_pred)
print(f"    Accuracy : {svm_acc*100:.2f}%   Time : {elapsed_svm:.1f}s")
save_cm(svm_cm, "SVM RBF — Confusion Matrix",
        f"{OUT}/10_svm_cm.png", "YlOrRd")


In [ ]:
# C sensitivity
print("  C sensitivity …")
C_vals, cva = [0.1, 1, 5, 10, 50], []
for c in C_vals:
    m = SVC(kernel="rbf", C=c, gamma="scale")
    m.fit(Xsvm_tr, y_svm_tr)
    cva.append(accuracy_score(y_sk_te, m.predict(Xsvm_te)))
    print(f"    C={c:5} → {cva[-1]*100:.2f}%")

fig, ax = plt.subplots(figsize=(10, 5))
_style(fig, ax)
ax.semilogx(C_vals, [a*100 for a in cva], "o-",
            color=RED, lw=2.5, ms=9)
ax.fill_between(C_vals, [a*100 for a in cva], alpha=0.15, color=RED)
ax.set_title("SVM — C Parameter Sensitivity", color="white", fontsize=13)
ax.set_xlabel("C (log scale)", color="#aaaaaa")
ax.set_ylabel("Accuracy (%)", color="#aaaaaa")
plt.tight_layout()
plt.savefig(f"{OUT}/11_svm_C.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()
print(f"  [✓] SVM plots saved")

In [ ]:
#  MODEL 5 — BASIC MLP
def build_basic_mlp():
    m = keras.Sequential([
        layers.Input(shape=(3072,)),
        layers.Dense(512, activation="relu"),
        layers.Dense(256, activation="relu"),
        layers.Dense(N_CLASSES, activation="softmax"),
    ], name="Basic_MLP")
    m.compile(optimizer="adam",
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m

mlp_b = build_basic_mlp()
mlp_b.summary()

print("\n  ▶ Basic MLP")
t0 = time.time()
h_mlp_b = mlp_b.fit(X_tr_flat, y_tr_oh,
                     epochs=40, batch_size=256,
                     validation_split=0.1,
                     callbacks=[EarlyStopping(patience=5,
                                              restore_best_weights=True)],
                     verbose=1)
elapsed_mlp_b = time.time() - t0
acc_mlp_b     = mlp_b.evaluate(X_te_flat, y_te_oh, verbose=0)[1]
results["Basic MLP"] = {"accuracy": acc_mlp_b, "train_time": elapsed_mlp_b,
                         "history": h_mlp_b.history, "type": "keras"}

In [ ]:
print(f"  Basic MLP Accuracy: {acc_mlp_b*100:.2f}%")

save_history(h_mlp_b, "Basic MLP", f"{OUT}/12_basic_mlp_hist.png")
pred_mlp_b = np.argmax(mlp_b.predict(X_te_flat, verbose=0), 1)
save_cm(confusion_matrix(y_te_raw, pred_mlp_b),
        "Basic MLP — Confusion Matrix",
        f"{OUT}/13_basic_mlp_cm.png", "Blues")

In [ ]:
# MODEL 6 — DEEP MLP with BatchNorm + Dropout
def build_deep_mlp():
    inp = layers.Input(shape=(3072,))
    x   = layers.Dense(1024, activation="relu")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.40)(x)
    x   = layers.Dense(512,  activation="relu")(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.35)(x)
    x   = layers.Dense(256,  activation="relu")(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.30)(x)
    x   = layers.Dense(128,  activation="relu")(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    m   = Model(inp, out, name="Deep_MLP")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m

mlp_d = build_deep_mlp()
mlp_d.summary()

In [ ]:
cbs_d = [EarlyStopping(patience=8, restore_best_weights=True),
         ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7, verbose=1)]


In [ ]:
print("\n   Deep MLP")
t0 = time.time()
h_mlp_d = mlp_d.fit(X_tr_flat, y_tr_oh,
                     epochs=80, batch_size=256,
                     validation_split=0.1,
                     callbacks=cbs_d, verbose=1)
elapsed_mlp_d = time.time() - t0
acc_mlp_d     = mlp_d.evaluate(X_te_flat, y_te_oh, verbose=0)[1]
results["Deep MLP (BN+Dropout)"] = {"accuracy": acc_mlp_d,
                                     "train_time": elapsed_mlp_d,
                                     "history": h_mlp_d.history,
                                     "type": "keras"}
print(f"  Deep MLP Accuracy: {acc_mlp_d*100:.2f}%")

save_history(h_mlp_d, "Deep MLP", f"{OUT}/14_deep_mlp_hist.png")
pred_mlp_d = np.argmax(mlp_d.predict(X_te_flat, verbose=0), 1)
save_cm(confusion_matrix(y_te_raw, pred_mlp_d),
        "Deep MLP — Confusion Matrix",
        f"{OUT}/15_deep_mlp_cm.png", "Purples")

In [ ]:
# Weight distribution after training
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
_style(fig, *axes)
dense_ls = [l for l in mlp_d.layers if isinstance(l, layers.Dense)]
for ax, dl in zip(axes, dense_ls[:4]):
    w = dl.get_weights()[0].flatten()
    ax.hist(w, bins=60, color=ACCENT, edgecolor="none", alpha=0.85)
    ax.set_title(f"{dl.name}\nweights", color="white", fontsize=10)
    ax.set_xlabel("Value", color="#aaaaaa")
    ax.set_ylabel("Count", color="#aaaaaa")
fig.suptitle("Deep MLP — Weight Distributions", color="white", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUT}/16_deep_mlp_weights.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()
print(f"  [✓] Deep MLP plots saved")

In [ ]:
#  MODEL 7 — SIMPLE CNN
def build_simple_cnn():
    inp = layers.Input(shape=(32, 32, 3))
    # Block 1
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)
    # Block 2
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)
    # Head
    x   = layers.Flatten()(x)
    x   = layers.Dense(512, activation="relu")(x)
    x   = layers.Dropout(0.50)(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    m   = Model(inp, out, name="Simple_CNN")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m

cnn_s = build_simple_cnn()
cnn_s.summary()

In [ ]:
cbs_s = [EarlyStopping(patience=8, restore_best_weights=True),
         ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7, verbose=1)]


In [ ]:
print("\n   Simple CNN")
t0 = time.time()
h_cnn_s = cnn_s.fit(X_tr_n, y_tr_oh,
                     epochs=60, batch_size=128,
                     validation_split=0.1,
                     callbacks=cbs_s, verbose=1)
elapsed_cnn_s = time.time() - t0
acc_cnn_s     = cnn_s.evaluate(X_te_n, y_te_oh, verbose=0)[1]
results["Simple CNN"] = {"accuracy": acc_cnn_s, "train_time": elapsed_cnn_s,
                          "history": h_cnn_s.history, "type": "keras"}
print(f"  Simple CNN Accuracy: {acc_cnn_s*100:.2f}%")

save_history(h_cnn_s, "Simple CNN", f"{OUT}/17_simple_cnn_hist.png")
pred_cnn_s = np.argmax(cnn_s.predict(X_te_n, verbose=0), 1)
save_cm(confusion_matrix(y_te_raw, pred_cnn_s),
        "Simple CNN — Confusion Matrix",
        f"{OUT}/18_simple_cnn_cm.png", "plasma")

In [ ]:
# Visualise conv1 filters
c1w = cnn_s.layers[1].get_weights()[0]   # (3,3,3,32)
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
_style(fig)
fig.suptitle("Simple CNN — Conv1 Filters", color="white", fontsize=14)
for i, ax in enumerate(axes.flat):
    if i < 32:
        filt = c1w[:, :, :, i]
        filt = (filt - filt.min()) / (filt.ptp() + 1e-8)
        ax.imshow(filt)
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT}/19_simple_cnn_filters.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()
print(f"  [✓] Simple CNN plots saved")

In [ ]:
# MODEL 8 — RESNET-STYLE CNN WITH RESIDUAL BLOCKS
def residual_block(x, filters, downsample=False):
    stride = 2 if downsample else 1
    shortcut = x
    # Main path
    y = layers.Conv2D(filters, 3, strides=stride, padding="same",
                      use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)
    y = layers.Conv2D(filters, 3, strides=1, padding="same",
                      use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    # Projection shortcut
    if downsample or x.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride,
                                 use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    y = layers.Add()([y, shortcut])
    y = layers.Activation("relu")(y)
    return y

def build_resnet_cnn():
    inp = layers.Input(shape=(32, 32, 3))
    x   = layers.Conv2D(64, 3, padding="same", use_bias=False)(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation("relu")(x)

    x   = residual_block(x, 64)
    x   = residual_block(x, 64)
    x   = residual_block(x, 128, downsample=True)
    x   = residual_block(x, 128)
    x   = residual_block(x, 256, downsample=True)
    x   = residual_block(x, 256)

    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(0.40)(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)

    m = Model(inp, out, name="ResNet_CNN")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m


In [ ]:
res_cnn = build_resnet_cnn()
res_cnn.summary()

cbs_res = [EarlyStopping(patience=10, restore_best_weights=True),
           ReduceLROnPlateau(factor=0.4, patience=4, min_lr=1e-8, verbose=1)]


In [ ]:
print("\n  ResNet-style CNN")
t0 = time.time()
h_res = res_cnn.fit(X_tr_n, y_tr_oh,
                    epochs=80, batch_size=128,
                    validation_split=0.1,
                    callbacks=cbs_res, verbose=1)
elapsed_res = time.time() - t0
acc_res     = res_cnn.evaluate(X_te_n, y_te_oh, verbose=0)[1]
results["ResNet-style CNN"] = {"accuracy": acc_res, "train_time": elapsed_res,
                                "history": h_res.history, "type": "keras"}
print(f"  ResNet CNN Accuracy: {acc_res*100:.2f}%")

In [ ]:
save_history(h_res, "ResNet-style CNN", f"{OUT}/20_resnet_hist.png")
pred_res = np.argmax(res_cnn.predict(X_te_n, verbose=0), 1)
save_cm(confusion_matrix(y_te_raw, pred_res),
        "ResNet CNN — Confusion Matrix",
        f"{OUT}/21_resnet_cm.png", "magma")

In [ ]:
#  Feature maps
feat_extractor = Model(inputs=res_cnn.input,
                        outputs=res_cnn.layers[4].output)   # after first BN
sample_img = X_te_n[0:1]
fmaps = feat_extractor.predict(sample_img, verbose=0)[0]  # (32,32,64)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
_style(fig)
fig.suptitle(f"ResNet CNN — Feature Maps  (true: {CLASS_NAMES[y_te_raw[0]]})",
             color="white", fontsize=13)
for i, ax in enumerate(axes.flat):
    if i < fmaps.shape[-1]:
        ax.imshow(fmaps[:,:,i], cmap="inferno")
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT}/22_resnet_fmaps.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()
print(f"  [✓] ResNet CNN plots saved")

In [ ]:
# MODEL 9 — ADVANCED CNN + DATA AUGMENTATION + COSINE LR SCHEDULE
datagen = ImageDataGenerator(
    rotation_range       = 15,
    width_shift_range    = 0.12,
    height_shift_range   = 0.12,
    horizontal_flip      = True,
    zoom_range           = 0.12,
    shear_range          = 0.08,
    channel_shift_range  = 0.08,
    fill_mode            = "reflect"
)
X_tr_aug = X_tr_n[:45_000]
y_tr_aug = y_tr_oh[:45_000]
X_val    = X_tr_n[45_000:]
y_val    = y_tr_oh[45_000:]
datagen.fit(X_tr_aug)
gen_tr = datagen.flow(X_tr_aug, y_tr_aug, batch_size=128, seed=SEED)


In [ ]:
# Augmentation showcase
batch_x, _ = next(gen_tr)
fig, axes  = plt.subplots(2, 10, figsize=(20, 5))
_style(fig)
fig.suptitle("Data Augmentation — Original vs Augmented",
             color="white", fontsize=13)
for i in range(10):
    orig = X_tr_aug[i]
    orig = (orig * ch_std + ch_mean)            # un-normalise for display
    orig = np.clip(orig, 0, 1)
    aug  = np.clip(batch_x[i] * ch_std + ch_mean, 0, 1)
    axes[0][i].imshow(orig);  axes[0][i].set_title("Orig",  color="#aaaaaa", fontsize=7)
    axes[1][i].imshow(aug);   axes[1][i].set_title("Augm",  color=GREEN,     fontsize=7)
    axes[0][i].axis("off");   axes[1][i].axis("off")
plt.tight_layout()
plt.savefig(f"{OUT}/23_augmentation.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()

In [ ]:
# Architecture
def se_block(x, ratio=16):
    """Squeeze-and-Excitation block for channel attention."""
    c   = x.shape[-1]
    se  = layers.GlobalAveragePooling2D()(x)
    se  = layers.Dense(max(c // ratio, 1), activation="relu")(se)
    se  = layers.Dense(c, activation="sigmoid")(se)
    se  = layers.Reshape((1, 1, c))(se)
    return layers.Multiply()([x, se])

def conv_bn_relu(x, filters, kernel=3, stride=1):
    x = layers.Conv2D(filters, kernel, strides=stride,
                      padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)

In [ ]:
def build_advanced_cnn():
    inp = layers.Input(shape=(32, 32, 3))

    # Stem
    x = conv_bn_relu(inp, 64)
    x = conv_bn_relu(x,  64)

    # Stage 1 — 128 filters
    res = layers.Conv2D(128, 1, use_bias=False)(x)
    res = layers.BatchNormalization()(res)
    x   = conv_bn_relu(x, 128)
    x   = conv_bn_relu(x, 128)
    x   = se_block(x)
    x   = layers.Add()([x, res])
    x   = layers.MaxPooling2D()(x)
    x   = layers.Dropout(0.20)(x)

    # Stage 2 — 256 filters
    res = layers.Conv2D(256, 1, use_bias=False)(x)
    res = layers.BatchNormalization()(res)
    x   = conv_bn_relu(x, 256)
    x   = conv_bn_relu(x, 256)
    x   = se_block(x)
    x   = layers.Add()([x, res])
    x   = layers.MaxPooling2D()(x)
    x   = layers.Dropout(0.25)(x)

     # Stage 3 — 512 filters
    res = layers.Conv2D(512, 1, use_bias=False)(x)
    res = layers.BatchNormalization()(res)
    x   = conv_bn_relu(x, 512)
    x   = conv_bn_relu(x, 512)
    x   = se_block(x)
    x   = layers.Add()([x, res])
    x   = layers.Dropout(0.30)(x)


    # Classifier head
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(512, activation="relu",
                        kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.45)(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)

    m = Model(inp, out, name="Advanced_CNN_SE")
    m.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return m

    
 
